# Deepwater Offshore Oil Production: Africa vs USA Gulf of Mexico — EDA

**Domain:** Oil and Gas  |  **Geography:** Gulf of Guinea / West-Central Africa vs United States (Gulf of Mexico proxy)  |  **Period:** 2010–2023

This notebook explores `data/raw/oil_production_merged.csv`, which merges:

* **Our World in Data energy dataset** — oil production (TWh/yr), per-capita production, GDP, population
  (https://github.com/owid/energy-data)
* **World Bank oil rents** — `NY.GDP.PETR.RT.ZS`, oil rents as % of GDP
  (https://api.worldbank.org/v2/country/{iso}/indicator/NY.GDP.PETR.RT.ZS?format=json)

> **Caveat:** OWID and the World Bank publish *national* totals. The USA series therefore includes onshore shale as well as
> deepwater Gulf of Mexico output; it is used as the GoM proxy. Cameroon and Ghana production is missing after 2016 in OWID,
> and World Bank oil rents end in 2021.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)

# Works whether the notebook is launched from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW = ROOT / "data" / "raw" / "oil_production_merged.csv"

df = pd.read_csv(RAW)
df["region"] = np.where(df["iso_code"] == "USA", "USA", "Africa")
# Project conversion (TWh/yr -> kbd as specified) and a physical barrel cross-check (1.7 MWh/bbl)
df["oil_production_kbd"] = df["oil_production_twh"] * 1000 / 8.76 / 24
df["oil_production_kbd_bbl"] = df["oil_production_twh"] * 1e6 / 1.7 / 365 / 1000
print(df.shape)
df.head()

## 1. Data quality overview
Null counts per column and coverage per country. Consumption columns are almost entirely empty for African producers in OWID, so the analysis focuses on production, rents, GDP and population.

In [ ]:
print("Rows:", len(df), "| Countries:", df["country"].nunique(), "| Years:", df["year"].min(), "-", df["year"].max())
print("Duplicate country-year keys:", df.duplicated(["iso_code", "year"]).sum())
display(df.isna().sum().to_frame("nulls"))
coverage = df.groupby("country").agg(
    years_with_production=("oil_production_twh", "count"),
    years_with_rents=("oil_rents_pct_gdp", "count"),
    peak_twh=("oil_production_twh", "max"),
)
coverage

## 2. Oil production by country, 2010–2023
Line chart in TWh/yr (log scale so the smaller Gulf of Guinea producers remain visible next to the USA).

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
sns.lineplot(data=df.dropna(subset=["oil_production_twh"]), x="year", y="oil_production_twh",
             hue="country", marker="o", ax=ax)
ax.set_yscale("log")
ax.set_title("Oil production by country, 2010–2023 (TWh/yr, log scale)")
ax.set_xlabel("Year"); ax.set_ylabel("Oil production (TWh/yr)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

change = df[df["year"].isin([2010, 2023])].pivot(index="country", columns="year", values="oil_production_twh")
change["pct_change_2010_2023"] = (change[2023] / change[2010] - 1) * 100
change.round(1)

**Finding:** The USA more than doubled output (≈3,874 → 9,648 TWh, about **+149%**), while every mature African deepwater producer declined:
Nigeria ≈ **−39%**, Angola ≈ **−37%**, Equatorial Guinea ≈ **−72%**. Congo (−11%) and Gabon (−4%) were comparatively stable.

## 3. Africa vs USA total production by year
To keep the Africa total comparable over time, only the five African producers with a complete 2010–2023 series are summed (Nigeria, Angola, Equatorial Guinea, Congo, Gabon).

In [ ]:
CORE5 = ["NGA", "AGO", "GNQ", "COG", "GAB"]
africa = df[df["iso_code"].isin(CORE5)].groupby("year")["oil_production_twh"].sum().rename("Africa (core 5)")
usa = df[df["iso_code"] == "USA"].set_index("year")["oil_production_twh"].rename("USA")
totals = pd.concat([africa, usa], axis=1)

ax = totals.plot(kind="bar", figsize=(13, 6), color=["#d95f02", "#1b9e77"], width=0.8)
ax.set_title("Africa (core 5 deepwater producers) vs USA — total oil production by year")
ax.set_xlabel("Year"); ax.set_ylabel("Oil production (TWh/yr)")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

totals["usa_to_africa_ratio"] = totals["USA"] / totals["Africa (core 5)"]
totals.round(2)

**Finding:** In 2010 the USA produced ~1.3× the combined output of the five African producers; by 2023 the ratio was ~5.2×. The African aggregate fell ~37% (≈2,943 → 1,851 TWh) as legacy deepwater fields in Angola and Nigeria matured faster than new projects came on stream.

## 4. Oil rents (% of GDP) heatmap
World Bank `NY.GDP.PETR.RT.ZS`; 2022–2023 are not yet published and are dropped.

In [ ]:
rents = df.pivot(index="country", columns="year", values="oil_rents_pct_gdp").dropna(axis=1, how="all")
plt.figure(figsize=(14, 5))
sns.heatmap(rents, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.5, cbar_kws={"label": "Oil rents, % of GDP"})
plt.title("Oil rents as % of GDP by country and year (World Bank)")
plt.xlabel("Year"); plt.ylabel("")
plt.tight_layout(); plt.show()

**Finding:** Congo, Angola, Gabon and Equatorial Guinea derived 30–49% of GDP from oil rents in 2010–2012. The 2014–2016 price collapse cut these shares by roughly two-thirds (e.g. Angola 38.6% in 2011 → 12.7% in 2015), with another dip in 2020.
US oil rents never exceed ~0.6% of GDP — the USA is a far larger producer but a vastly more diversified economy, so price shocks are a fiscal risk mainly for the African producers.

## 5. Oil production vs GDP
Interactive Plotly scatter (log–log), one point per country-year, sized by population. GDP is OWID's series (constant international dollars).

In [ ]:
scatter_df = df.dropna(subset=["oil_production_twh", "gdp"]).copy()
fig = px.scatter(
    scatter_df, x="gdp", y="oil_production_twh", color="country", size="population",
    hover_data=["year", "oil_rents_pct_gdp"], log_x=True, log_y=True, size_max=45,
    title="Oil production vs GDP (2010–2023, log–log, bubble = population)",
    labels={"gdp": "GDP (international $)", "oil_production_twh": "Oil production (TWh/yr)"},
)
fig.update_layout(height=550)
fig.show()

corr = scatter_df.groupby("country")[["oil_production_twh", "gdp"]].apply(lambda g: g["oil_production_twh"].corr(g["gdp"])).rename("r(production, GDP)")
corr.round(2).sort_values()

**Finding:** In the USA production and GDP rise together (r ≈ 0.97). Nigeria (r ≈ −0.88), Gabon (≈ −0.76) and Congo (≈ −0.62) show the opposite: GDP kept growing while oil output fell or stagnated, i.e. growth is decoupling from crude volumes. Angola is essentially uncorrelated (≈ −0.06). Equatorial Guinea (≈ 0.96) is the cautionary case — GDP has shrunk in step with its collapsing production.

## 6. Stacked area — composition of African production
All seven African producers stacked (Cameroon and Ghana only through 2016, where OWID coverage ends).

In [ ]:
afr = (df[df["region"] == "Africa"]
       .pivot(index="year", columns="country", values="oil_production_twh")
       .fillna(0))
afr = afr[afr.sum().sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(13, 6))
ax.stackplot(afr.index, afr.T.values, labels=afr.columns, alpha=0.85,
             colors=sns.color_palette("tab10", len(afr.columns)))
ax.set_title("African deepwater producers — stacked oil production (TWh/yr)")
ax.set_xlabel("Year"); ax.set_ylabel("Oil production (TWh/yr)")
ax.legend(loc="upper right", ncol=2)
ax.set_xlim(afr.index.min(), afr.index.max())
plt.tight_layout(); plt.show()

share = afr.div(afr.sum(axis=1), axis=0) * 100
share.loc[[2010, 2016, 2023]].round(1)

**Finding:** Nigeria and Angola together supply ~80% of the group's output throughout. Angola's share is resilient, but both have lost about a third of their volume; Equatorial Guinea's contraction (−72%) is the steepest in the sample.

## 7. Bonus — unit conversion check
The project spec converts TWh/yr with `twh * 1000 / 8.76 / 24`. A physical barrel holds ≈1.7 MWh, so the barrel-based figure is also shown for context.

In [ ]:
conv = df[df["year"] == 2023][["country", "oil_production_twh", "oil_production_kbd", "oil_production_kbd_bbl"]]
conv.dropna().sort_values("oil_production_twh", ascending=False).round(1)

## 8. Summary of findings

1. **Divergence:** USA output grew ~149% (2010→2023) while the five core African deepwater producers fell ~37% combined.
2. **Scale gap:** the USA:Africa production ratio widened from ~1.3× to ~5.2×.
3. **Maturing basins:** Nigeria (−39%), Angola (−37%) and Equatorial Guinea (−72%) show classic post-plateau deepwater decline; Congo and Gabon held roughly flat.
4. **Fiscal exposure:** oil rents reached 30–49% of GDP in Congo, Angola, Gabon and Equatorial Guinea, vs <1% in the USA, and halved or worse during the 2015–2016 and 2020 price shocks.
5. **Decoupling vs dependence:** GDP is negatively correlated with output in Nigeria, Gabon and Congo (growth decoupling from crude), but tracks production closely in Equatorial Guinea (r ≈ 0.96), the most oil-dependent and fastest-declining producer.

*Data limitations:* national totals rather than basin-level offshore volumes; OWID gaps for Cameroon/Ghana after 2016; World Bank rents end in 2021.